## 准备数据

In [6]:
import os
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, optimizers, datasets

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'  # or any {'0', '1', '2'}

def mnist_dataset():
    (x, y), (x_test, y_test) = datasets.mnist.load_data()
    #normalize
    x = x/255.0
    x_test = x_test/255.0
    
    return (x, y), (x_test, y_test)

In [7]:
print(list(zip([1, 2, 3, 4], ['a', 'b', 'c', 'd'])))

[(1, 'a'), (2, 'b'), (3, 'c'), (4, 'd')]


## 建立模型

In [8]:
class myModel:
    def __init__(self):
        ####################
        '''声明模型对应的参数'''
        ####################
        self.W1 = tf.Variable(tf.random.normal([784, 512], stddev=0.1))
        self.b1 = tf.Variable(tf.zeros([512]))
        self.W2 = tf.Variable(tf.random.normal([512, 10], stddev=0.1))
        self.b2 = tf.Variable(tf.zeros([10]))
        
    def __call__(self, x):
        ####################
        '''实现模型函数体，返回未归一化的logits'''
        ####################
        x = tf.reshape(x, [-1, 784])
        h = tf.nn.relu(tf.matmul(x, self.W1) + self.b1)
        logits = tf.matmul(h, self.W2) + self.b2
        return logits
        
model = myModel()

optimizer = optimizers.Adam()

## 计算 loss

In [9]:
@tf.function
def compute_loss(logits, labels):
    return tf.reduce_mean(
        tf.nn.sparse_softmax_cross_entropy_with_logits(
            logits=logits, labels=labels))

@tf.function
def compute_accuracy(logits, labels):
    predictions = tf.argmax(logits, axis=1)
    return tf.reduce_mean(tf.cast(tf.equal(predictions, labels), tf.float32))

@tf.function
def train_one_step(model, optimizer, x, y):
    with tf.GradientTape() as tape:
        logits = model(x)
        loss = compute_loss(logits, y)

    # compute gradient
    trainable_vars = [model.W1, model.W2, model.b1, model.b2]
    grads = tape.gradient(loss, trainable_vars)
    for g, v in zip(grads, trainable_vars):
        v.assign_sub(0.01*g)

    accuracy = compute_accuracy(logits, y)

    # loss and accuracy is scalar tensor
    return loss, accuracy

@tf.function
def test(model, x, y):
    logits = model(x)
    loss = compute_loss(logits, y)
    accuracy = compute_accuracy(logits, y)
    return loss, accuracy

## 实际训练

In [10]:
train_data, test_data = mnist_dataset()
for epoch in range(200):
    loss, accuracy = train_one_step(model, optimizer, 
                                    tf.constant(train_data[0], dtype=tf.float32), 
                                    tf.constant(train_data[1], dtype=tf.int64))
    print('epoch', epoch, ': loss', loss.numpy(), '; accuracy', accuracy.numpy())
loss, accuracy = test(model, 
                      tf.constant(test_data[0], dtype=tf.float32), 
                      tf.constant(test_data[1], dtype=tf.int64))

print('test loss', loss.numpy(), '; accuracy', accuracy.numpy())

epoch 0 : loss 3.4196875 ; accuracy 0.13315
epoch 1 : loss 3.1219506 ; accuracy 0.14366667
epoch 2 : loss 2.9295588 ; accuracy 0.14976667
epoch 3 : loss 2.79326 ; accuracy 0.15685
epoch 4 : loss 2.6896935 ; accuracy 0.16486667
epoch 5 : loss 2.6067328 ; accuracy 0.17316666
epoch 6 : loss 2.5376573 ; accuracy 0.1812
epoch 7 : loss 2.478433 ; accuracy 0.18941666
epoch 8 : loss 2.4264264 ; accuracy 0.19793333
epoch 9 : loss 2.3798094 ; accuracy 0.2068
epoch 10 : loss 2.3372757 ; accuracy 0.21596667
epoch 11 : loss 2.2978823 ; accuracy 0.2248
epoch 12 : loss 2.260936 ; accuracy 0.23485
epoch 13 : loss 2.2259357 ; accuracy 0.2447
epoch 14 : loss 2.1925192 ; accuracy 0.25426668
epoch 15 : loss 2.160427 ; accuracy 0.26463333
epoch 16 : loss 2.1294718 ; accuracy 0.27416667
epoch 17 : loss 2.0995188 ; accuracy 0.2849
epoch 18 : loss 2.0704741 ; accuracy 0.2957
epoch 19 : loss 2.0422647 ; accuracy 0.30556667
epoch 20 : loss 2.0148342 ; accuracy 0.31661665
epoch 21 : loss 1.9881393 ; accuracy 0.3